In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 100)

# Plot settings
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

print("Libraries loaded successfully")

Libraries loaded successfully


In [15]:
# Load datasets - using more rows this time
print("Loading datasets...")

# Load comments (100k rows for preprocessing)
comments_path = Path('../data/raw/digikala-comments.csv')
comments_df = pd.read_csv(comments_path, nrows=100000)
print(f"Comments loaded: {comments_df.shape}")

# Load products (100k rows for preprocessing)
products_path = Path('../data/raw/digikala-products.csv')
products_df = pd.read_csv(products_path, nrows=100000)
print(f"Products loaded: {products_df.shape}")

print("\nDatasets loaded successfully!")

Loading datasets...
Comments loaded: (100000, 15)
Products loaded: (100000, 12)

Datasets loaded successfully!


In [16]:
# Check common keys for merging
print("Checking merge keys...")
print(f"\nUnique product_id in comments: {comments_df['product_id'].nunique():,}")
print(f"Unique id in products: {products_df['id'].nunique():,}")

# Check how many comments have matching products
matching_products = comments_df['product_id'].isin(products_df['id']).sum()
print(f"\nComments with matching products: {matching_products:,} ({matching_products/len(comments_df)*100:.2f}%)")

# Check sample of product_ids
print("\nSample product_ids from comments:")
print(comments_df['product_id'].head(10).tolist())
print("\nSample ids from products:")
print(products_df['id'].head(10).tolist())

Checking merge keys...

Unique product_id in comments: 9,937
Unique id in products: 77,559

Comments with matching products: 12,691 (12.69%)

Sample product_ids from comments:
[252058, 252058, 3331597, 3331329, 3255700, 3305270, 3480048, 3480048, 7626372, 821812]

Sample ids from products:
[7096438, 2845119, 6117745, 1912926, 6335462, 6335562, 7096725, 7096768, 6005981, 4077287]


In [17]:
# Merge datasets
print("Merging datasets...")

# Left join: keep all comments, add product info
merged_df = comments_df.merge(
    products_df,
    left_on='product_id',
    right_on='id',
    how='left',
    suffixes=('_comment', '_product')
)

print(f"\nMerged dataset shape: {merged_df.shape}")
print(f"Columns: {merged_df.columns.tolist()}")

# Check merge success
print(f"\nRows with product info: {merged_df['title_fa'].notna().sum():,}")
print(f"Rows without product info: {merged_df['title_fa'].isna().sum():,}")

Merging datasets...

Merged dataset shape: (100554, 27)
Columns: ['id_comment', 'title', 'body', 'created_at', 'rate', 'recommendation_status', 'is_buyer', 'product_id', 'advantages', 'disadvantages', 'likes', 'dislikes', 'seller_title', 'seller_code', 'true_to_size_rate', 'id_product', 'title_fa', 'Rate', 'Rate_cnt', 'Category1', 'Category2', 'Brand', 'Price', 'Seller', 'Is_Fake', 'min_price_last_month', 'sub_category']

Rows with product info: 13,245
Rows without product info: 87,309


In [18]:
# Analyze comments with/without product info
print("Analyzing matched vs unmatched comments...\n")

# Check if matched comments have different characteristics
matched_comments = merged_df[merged_df['title_fa'].notna()]
unmatched_comments = merged_df[merged_df['title_fa'].isna()]

print(f"Matched comments: {len(matched_comments):,}")
print(f"  - Average rate: {matched_comments['rate'].mean():.2f}")
print(f"  - Buyers: {matched_comments['is_buyer'].sum():,} ({matched_comments['is_buyer'].sum()/len(matched_comments)*100:.1f}%)")

print(f"\nUnmatched comments: {len(unmatched_comments):,}")
print(f"  - Average rate: {unmatched_comments['rate'].mean():.2f}")
print(f"  - Buyers: {unmatched_comments['is_buyer'].sum():,} ({unmatched_comments['is_buyer'].sum()/len(unmatched_comments)*100:.1f}%)")

# Check categories of matched products
print(f"\nCategories in matched products:")
print(matched_comments['Category1'].value_counts())

Task was destroyed but it is pending!
task: <Task pending name='Task-135' coro=<_async_in_context.<locals>.run_in_context_pre311() done, defined at C:\Users\ASUS\Desktop\digikala-prods-comments-analysis\venv\lib\site-packages\ipykernel\utils.py:76> wait_for=<Task pending name='Task-136' coro=<_async_in_context.<locals>.preserve_context() running at C:\Users\ASUS\Desktop\digikala-prods-comments-analysis\venv\lib\site-packages\ipykernel\utils.py:68> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Users\ASUS\Desktop\digikala-prods-comments-analysis\venv\lib\site-packages\zmq\eventloop\zmqstream.py:563]>
C:\Users\ASUS\Desktop\digikala-prods-comments-analysis\venv\lib\site-packages\IPython\core\compilerop.py:86: RuntimeWarning: coroutine '_async_in_context.<locals>.preserve_context' was never awaited
  return compile(source, filename, symbol, self.flags | PyCF_ONLY_AST, 1)
Task was destroyed but it is pending!
task: <Task pending name='Task-136' coro=<_async

Analyzing matched vs unmatched comments...

Matched comments: 13,245
  - Average rate: 3.71
  - Buyers: 12,785 (96.5%)

Unmatched comments: 87,309
  - Average rate: 3.62
  - Buyers: 83,267 (95.4%)

Categories in matched products:
Category1
آرایش مو                      6206
آرایش لب                      2686
آرایش چشم                     1774
آرایش صورت                    1693
آرایش ابرو                     851
ابزار مراقبت پا                 16
آموزش زبان                      11
ابزار توانمند سازی               4
آموزش موسیقی                     3
آموزش نرم‌افزار و کامپیوتر       1
Name: count, dtype: int64


In [19]:
# Load only product_id column from full comments dataset
print("Loading unique product IDs from full comments dataset...")
comments_product_ids = pd.read_csv(comments_path, usecols=['product_id'])
unique_product_ids = comments_product_ids['product_id'].unique()

print(f"Total unique products in comments: {len(unique_product_ids):,}")
print(f"Memory usage: {comments_product_ids.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

del comments_product_ids  # Free memory


Loading unique product IDs from full comments dataset...
Total unique products in comments: 331,599
Memory usage: 46.97 MB


In [50]:
# Filter products dataset to only include products that have comments
print("Loading products dataset...")
products_df = pd.read_csv(products_path, encoding='utf-8-sig')
print(f"Total products in dataset: {len(products_df):,}")

# Filter to only products with comments
filtered_products = products_df[products_df['id'].isin(unique_product_ids)]
print(f"Products with comments: {len(filtered_products):,}")
print(f"Match rate: {len(filtered_products)/len(unique_product_ids)*100:.2f}%")

Loading products dataset...


C:\Users\ASUS\AppData\Local\Temp\ipykernel_16508\1826733924.py:3: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  products_df = pd.read_csv(products_path, encoding='utf-8-sig')


Total products in dataset: 1,283,496
Products with comments: 454,969
Match rate: 137.20%


In [51]:
import os

# Create processed directory if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)

# Save filtered products for faster loading later
filtered_products.to_csv('../data/processed/products_with_comments.csv', index=False, encoding='utf-8-sig')
print(f"Saved to: data/processed/products_with_comments.csv")
print(f"Size: {filtered_products.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Saved to: data/processed/products_with_comments.csv
Size: 298.60 MB


In [52]:
# Check for duplicate product_ids
duplicate_count = products_df['id'].duplicated().sum()
print(f"Duplicate product_ids: {duplicate_count}")

# See some examples
if duplicate_count > 0:
    duplicates = products_df[products_df['id'].duplicated(keep=False)]
    print(f"\nSample duplicates:")
    print(duplicates[['id', 'title_fa']].head(10))


Duplicate product_ids: 335144

Sample duplicates:
           id                                                       title_fa
25    1452939                                  آبسلانگ مدل 1002 بسته 30 عددی
26    1452939                                  آبسلانگ مدل 1002 بسته 30 عددی
72   10420922                             مداد ابرو پیپا مدل پرفکت شماره 104
73   10420922                             مداد ابرو پیپا مدل پرفکت شماره 104
137   9803517                                       مداد ابرو ایفسن شماره 03
138   9803517                                       مداد ابرو ایفسن شماره 03
281   3464108  رنگ ابرو آتوسا رویال شماره 4 حجم 15 میلی لیتر رنگ بلوطی متوسط
282   3464168          رنگ ابرو آتوسا رویال شماره 6 حجم 15 میلی لیتر رنگ شنی
283   3464108  رنگ ابرو آتوسا رویال شماره 4 حجم 15 میلی لیتر رنگ بلوطی متوسط
284   3464168          رنگ ابرو آتوسا رویال شماره 6 حجم 15 میلی لیتر رنگ شنی


In [53]:
# Check if duplicates have different data
duplicate_ids = products_df[products_df.duplicated(subset=['id'], keep=False)]['id'].unique()

# Look at first duplicate group in detail
first_dup_id = duplicate_ids[0]
dup_group = products_df[products_df['id'] == first_dup_id]

print(f"Checking product_id: {first_dup_id}")
print("\nAll columns for this duplicate:")
print(dup_group.to_string())

# Check if ALL columns are identical
print("\n\nAre all rows completely identical?")
print(dup_group.duplicated(keep=False).all())


Checking product_id: 1452939

All columns for this duplicate:
         id                       title_fa  Rate  Rate_cnt Category1 Category2   Brand   Price         Seller  Is_Fake  min_price_last_month sub_category
25  1452939  آبسلانگ مدل 1002 بسته 30 عددی    76       166   آبسلانگ       NaN  متفرقه  245000  سلامت ساز راد    False                     0       beauty
26  1452939  آبسلانگ مدل 1002 بسته 30 عددی    76       166   آبسلانگ       NaN  متفرقه  245000  سلامت ساز راد    False                     0       beauty


Are all rows completely identical?
True


In [47]:
# Check a few more duplicate groups
import random

sample_dup_ids = random.sample(list(duplicate_ids), min(5, len(duplicate_ids)))

for dup_id in sample_dup_ids:
    dup_group = products_df[products_df['id'] == dup_id]
    all_identical = dup_group.duplicated(keep=False).all()
    
    print(f"Product ID: {dup_id} | Count: {len(dup_group)} | All identical: {all_identical}")
    
    if not all_identical:
        print("  ⚠️ This one has differences!")
        print(dup_group.to_string())
        print("\n")

Product ID: 11698541 | Count: 2 | All identical: True
Product ID: 4845217 | Count: 2 | All identical: True
Product ID: 7539426 | Count: 3 | All identical: True
Product ID: 9907343 | Count: 2 | All identical: True
Product ID: 9728012 | Count: 2 | All identical: True


In [49]:
# Find duplicates that are NOT identical
non_identical_dups = []

for dup_id in duplicate_ids:
    dup_group = products_df[products_df['id'] == dup_id]
    if not dup_group.duplicated(keep=False).all():
        non_identical_dups.append(dup_id)

print(f"Total duplicate IDs: {len(duplicate_ids):,}")
print(f"Non-identical duplicates: {len(non_identical_dups):,}")
print(f"Percentage: {len(non_identical_dups)/len(duplicate_ids)*100:.2f}%")

Total duplicate IDs: 258,718
Non-identical duplicates: 10,920
Percentage: 4.22%


In [54]:
# Check which columns differ in non-identical duplicates
if len(non_identical_dups) > 0:
    sample_check = non_identical_dups[:20]  # Check first 20
    
    diff_columns = {}
    
    for dup_id in sample_check:
        dup_group = products_df[products_df['id'] == dup_id]
        
        for col in dup_group.columns:
            if col != 'id' and dup_group[col].nunique() > 1:
                diff_columns[col] = diff_columns.get(col, 0) + 1
    
    print("\nColumns with differences:")
    for col, count in sorted(diff_columns.items(), key=lambda x: x[1], reverse=True):
        print(f"  {col}: {count} cases")



Columns with differences:
  Price: 13 cases
  Seller: 8 cases
  Rate_cnt: 5 cases
  min_price_last_month: 2 cases


In [55]:
# Keep the most recent/complete record for each product_id
# Priority: keep rows with more complete data and higher Rate_cnt

filtered_products_clean = filtered_products.sort_values(
    ['Rate_cnt', 'Price'],  # Sort by Rate_cnt (more reviews = more reliable), then Price
    ascending=[False, True]  # Higher Rate_cnt first, lower Price first
).drop_duplicates(
    subset='id',
    keep='first'  # Keep the first occurrence after sorting
)

print(f"Original: {len(filtered_products):,} rows")
print(f"After deduplication: {len(filtered_products_clean):,} rows")
print(f"Removed: {len(filtered_products) - len(filtered_products_clean):,} duplicates")

# Save the cleaned version
filtered_products_clean.to_csv('../data/processed/products_with_comments_clean.csv', 
                               index=False, 
                               encoding='utf-8-sig')


Original: 454,969 rows
After deduplication: 331,599 rows
Removed: 123,370 duplicates


In [61]:
# Load 100k comments
print("Loading 100k comments...")
comments = pd.read_csv('../data/raw/digikala-comments.csv', 
                       encoding='utf-8-sig',
                       nrows=100000)
print(f"Comments loaded: {len(comments):,}")

# Get unique product_ids from these 100k comments
unique_product_ids = comments['product_id'].unique()
print(f"Unique product_ids in 100k comments: {len(unique_product_ids):,}")

# Load 100k products and filter
print("\nLoading 100k products and filtering...")
products = pd.read_csv('../data/raw/digikala-products.csv', 
                       encoding='utf-8-sig',
                       nrows=100000)
filtered_products = products[products['id'].isin(unique_product_ids)].copy()
print(f"Filtered products: {len(filtered_products):,}")

# Remove duplicates
filtered_products_clean = filtered_products.sort_values(
    ['Rate_cnt', 'Price'], 
    ascending=[False, True]
).drop_duplicates(subset=['id'], keep='first')
print(f"Products after removing duplicates: {len(filtered_products_clean):,}")

# Merge
merged_data = comments.merge(
    filtered_products_clean,
    left_on='product_id',
    right_on='id',
    how='left',
    suffixes=('_comment', '_product')
)

print(f"\nMerged shape: {merged_data.shape}")
print(f"Match rate: {merged_data['id_product'].notna().sum()/len(merged_data)*100:.2f}%")

# Save
merged_data.to_csv('../data/processed/merged_100k.csv', 
                   index=False, 
                   encoding='utf-8-sig')
print("\nSaved to: data/processed/merged_100k.csv")


Loading 100k comments...
Comments loaded: 100,000
Unique product_ids in 100k comments: 9,937

Loading 100k products and filtering...
Filtered products: 1,056
Products after removing duplicates: 996

Merged shape: (100000, 27)
Match rate: 12.69%

Saved to: data/processed/merged_100k.csv
